# MedGuard AI: Evidence-Constrained Clinical Prescription Review Copilot

This notebook builds a hackathon-ready clinical review agent using:

- **ROCm + vLLM** for serving Qwen3 on AMD GPUs
- **Qwen3** for evidence-grounded reasoning and intervention drafting
- **PydanticAI** for typed agent orchestration and tool calling
- **FHIR-style synthetic patient data** for a US healthcare workflow

The goal is to review a proposed prescription against medication history, medical history, allergies, labs, drug interactions, DUR rules, and controlled-substance risk.

This tutorial includes the following sections:

- [1. Launching the vLLM server](#step1)
- [2. Installing dependencies](#step2)
- [3. Connecting PydanticAI to Qwen3](#step3)
- [4. Defining clinical review schemas](#step4)
- [5. Creating synthetic FHIR-style patient data](#step5)
- [6. Building DUR and evidence tools](#step6)
- [7. Creating the MedGuard AI agent](#step7)
- [8. Running a clinical prescription review](#step8)
- [9. Evaluating the solution](#step9)
- [10. Challenge ideas for the hackathon](#step10)

> Safety note: this is a clinical decision-support prototype for synthetic data. It must not be used to make autonomous medication decisions.

<a id="step1"></a>

## Step 1: Launching the vLLM server

Start a vLLM OpenAI-compatible endpoint for Qwen3 on an AMD GPU with ROCm. Open a terminal in your Jupyter environment and run one of the commands below.

For a strong hackathon demo, use `Qwen/Qwen3-30B-A3B` if your GPU memory allows it. For a lighter setup, use `Qwen/Qwen3-14B` or `Qwen/Qwen3-8B`.

> **Copy and paste this command to launch your vLLM server:**

```bash
vllm serve Qwen/Qwen3-30B-A3B \
  --host 0.0.0.0 \
  --port 8000 \
  --served-model-name Qwen3-30B-A3B \
  --trust-remote-code \
  --max-model-len 32768 \
  --gpu-memory-utilization 0.90
```

If you need a smaller model:

```bash
vllm serve Qwen/Qwen3-14B \
  --host 0.0.0.0 \
  --port 8000 \
  --served-model-name Qwen3-14B \
  --trust-remote-code
```

In [ ]:
import os

BASE_URL = "http://localhost:8000/v1"
MODEL_NAME = os.getenv("MODEL_NAME", "Qwen3-30B-A3B")

os.environ["BASE_URL"] = BASE_URL
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "abc-123")

print("Config set")
print("BASE_URL:", BASE_URL)
print("MODEL_NAME:", MODEL_NAME)

Verify the model is available. If this cell fails, keep going: the deterministic clinical-review engine later in the notebook works without a running LLM server.

In [ ]:
!curl http://localhost:8000/v1/models -H "Authorization: Bearer $OPENAI_API_KEY"

<a id="step2"></a>

## Step 2: Installing dependencies

The notebook uses PydanticAI for the agent, Pydantic for typed outputs, and Rich for readable display.

In [ ]:
!pip install -q pydantic pydantic_ai openai rich

<a id="step3"></a>

## Step 3: Connecting PydanticAI to Qwen3

This mirrors the sample notebook pattern: create an OpenAI-compatible provider pointed at the vLLM endpoint, then create a PydanticAI agent model.

In [ ]:
try:
    from pydantic_ai.models.openai import OpenAIModel
    from pydantic_ai.providers.openai import OpenAIProvider

    provider = OpenAIProvider(
        base_url=os.environ["BASE_URL"],
        api_key=os.environ["OPENAI_API_KEY"],
    )

    agent_model = OpenAIModel(os.environ.get("MODEL_NAME", MODEL_NAME), provider=provider)
    print("PydanticAI model configured:", MODEL_NAME)
except Exception as exc:
    agent_model = None
    print("PydanticAI model was not configured yet.")
    print("Run the dependency install cell and make sure your vLLM server is available.")
    print(type(exc).__name__, str(exc)[:300])

In [ ]:
import asyncio

async def run_async(prompt: str):
    if agent is None:
        raise RuntimeError("MedGuard agent is not configured. Install pydantic_ai and configure the vLLM endpoint.")
    result = await agent.run(prompt)
    return result.output

<a id="step4"></a>

## Step 4: Defining clinical review schemas

Clinical review needs structure. The agent will return a typed decision with findings, citations, and intervention text.

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field


Severity = Literal["low", "moderate", "high", "critical"]
Action = Literal["approve", "monitor", "clarify", "substitute", "block"]


class ProposedPrescription(BaseModel):
    drug_name: str
    dose: str
    route: str
    frequency: str
    duration: str
    indication: str


class EvidenceCitation(BaseModel):
    source: str
    title: str
    url: str
    note: str


class Finding(BaseModel):
    category: Literal[
        "drug_interaction",
        "allergy",
        "contraindication",
        "duplicate_therapy",
        "dose_adjustment",
        "monitoring",
        "controlled_substance",
        "missing_data",
    ]
    severity: Severity
    title: str
    rationale: str
    recommendation: str
    evidence_ids: list[str] = Field(default_factory=list)


class ClinicalReviewDecision(BaseModel):
    patient_id: str
    prescription: ProposedPrescription
    overall_risk: Severity
    recommended_action: Action
    key_findings: list[Finding]
    missing_information: list[str] = Field(default_factory=list)
    evidence: list[EvidenceCitation] = Field(default_factory=list)
    clinician_message: str
    pharmacist_note: str

<a id="step5"></a>

## Step 5: Creating synthetic FHIR-style patient data

For a hackathon, avoid real PHI. Use synthetic FHIR-like records generated by Synthea or created manually as below.

This case is intentionally high signal:

- Older adult with atrial fibrillation on warfarin
- CKD stage 3b with elevated potassium
- Documented sulfonamide allergy
- Existing opioid and benzodiazepine overlap
- Proposed prescription: TMP-SMX for UTI

In [ ]:
from pprint import pprint

PATIENTS = {
    "patient-001": {
        "resourceType": "Bundle",
        "patient": {
            "id": "patient-001",
            "name": "Synthetic Patient",
            "age": 74,
            "sex": "female",
        },
        "conditions": [
            "Atrial fibrillation",
            "Chronic kidney disease stage 3b",
            "Type 2 diabetes mellitus",
            "Hypertension",
            "Obstructive sleep apnea",
        ],
        "allergies": [
            {
                "substance": "sulfonamide antibiotics",
                "reaction": "anaphylaxis",
                "severity": "severe",
            },
            {
                "substance": "penicillin",
                "reaction": "rash",
                "severity": "moderate",
            },
        ],
        "medications": [
            {"name": "warfarin", "dose": "5 mg", "route": "oral", "frequency": "daily"},
            {"name": "lisinopril", "dose": "20 mg", "route": "oral", "frequency": "daily"},
            {"name": "spironolactone", "dose": "25 mg", "route": "oral", "frequency": "daily"},
            {"name": "metformin", "dose": "1000 mg", "route": "oral", "frequency": "twice daily"},
            {"name": "alprazolam", "dose": "0.5 mg", "route": "oral", "frequency": "as needed"},
            {"name": "oxycodone", "dose": "5 mg", "route": "oral", "frequency": "every 6 hours as needed"},
        ],
        "observations": {
            "eGFR": {"value": 38, "unit": "mL/min/1.73m2"},
            "potassium": {"value": 5.3, "unit": "mmol/L"},
            "INR": {"value": 3.2, "unit": "ratio"},
            "AST": {"value": 32, "unit": "U/L"},
            "ALT": {"value": 29, "unit": "U/L"},
        },
    }
}

proposed_rx = ProposedPrescription(
    drug_name="sulfamethoxazole-trimethoprim DS",
    dose="800 mg / 160 mg",
    route="oral",
    frequency="twice daily",
    duration="7 days",
    indication="uncomplicated urinary tract infection",
)

pprint(PATIENTS["patient-001"])
proposed_rx

<a id="step6"></a>

## Step 6: Building DUR and evidence tools

The key design principle is: deterministic tools identify safety facts, and the LLM explains them with citations. This reduces hallucination risk and makes the system more enterprise-ready.

In [ ]:
DRUG_DB = {
    "warfarin": {
        "ingredients": ["warfarin"],
        "classes": ["anticoagulant", "vitamin_k_antagonist"],
        "controlled": False,
    },
    "sulfamethoxazole-trimethoprim ds": {
        "ingredients": ["sulfamethoxazole", "trimethoprim"],
        "classes": ["sulfonamide_antibiotic", "folate_antagonist_antibiotic"],
        "controlled": False,
    },
    "tmp-smx": {
        "ingredients": ["sulfamethoxazole", "trimethoprim"],
        "classes": ["sulfonamide_antibiotic", "folate_antagonist_antibiotic"],
        "controlled": False,
    },
    "lisinopril": {
        "ingredients": ["lisinopril"],
        "classes": ["ace_inhibitor"],
        "controlled": False,
    },
    "spironolactone": {
        "ingredients": ["spironolactone"],
        "classes": ["potassium_sparing_diuretic", "aldosterone_antagonist"],
        "controlled": False,
    },
    "metformin": {
        "ingredients": ["metformin"],
        "classes": ["biguanide"],
        "controlled": False,
    },
    "alprazolam": {
        "ingredients": ["alprazolam"],
        "classes": ["benzodiazepine"],
        "controlled": True,
        "schedule": "C-IV",
    },
    "oxycodone": {
        "ingredients": ["oxycodone"],
        "classes": ["opioid_analgesic"],
        "controlled": True,
        "schedule": "C-II",
        "mme_factor": 1.5,
    },
}


EVIDENCE_KB = {
    "dailyMed_tmp_smx": {
        "source": "DailyMed",
        "title": "Sulfamethoxazole and Trimethoprim labeling",
        "url": "https://dailymed.nlm.nih.gov/dailymed/",
        "note": "FDA labeling source for indications, warnings, contraindications, and adverse reactions.",
    },
    "dailyMed_warfarin": {
        "source": "DailyMed",
        "title": "Warfarin labeling",
        "url": "https://dailymed.nlm.nih.gov/dailymed/",
        "note": "FDA labeling source for warfarin interaction and bleeding-risk warnings.",
    },
    "rxnorm": {
        "source": "NLM RxNorm/RxNav",
        "title": "RxNorm and RxNav APIs",
        "url": "https://lhncbc.nlm.nih.gov/RxNav/APIs/index.html",
        "note": "Medication normalization and drug concept mapping.",
    },
    "cdc_opioid": {
        "source": "CDC",
        "title": "CDC opioid prescribing clinical guidance",
        "url": "https://www.cdc.gov/overdose-prevention/hcp/clinical-guidance/index.html",
        "note": "Guidance for opioid risk review; does not replace clinical judgment.",
    },
    "fhir_medication_request": {
        "source": "HL7 FHIR",
        "title": "FHIR MedicationRequest",
        "url": "https://hl7.org/fhir/R4/medicationrequest.html",
        "note": "FHIR resource for medication orders or requests.",
    },
}


INTERACTION_RULES = [
    {
        "pair": {"warfarin", "sulfamethoxazole"},
        "severity": "critical",
        "title": "TMP-SMX may increase warfarin anticoagulation effect",
        "rationale": "Patient is already anticoagulated and has an INR of 3.2. TMP-SMX is a high-risk interacting antibiotic in a warfarin patient.",
        "recommendation": "Block pending clinician/pharmacist review. Consider an alternative antibiotic or define an INR monitoring and dose-adjustment plan.",
        "evidence_ids": ["dailyMed_tmp_smx", "dailyMed_warfarin"],
    },
    {
        "pair": {"lisinopril", "spironolactone"},
        "severity": "high",
        "title": "ACE inhibitor plus potassium-sparing diuretic with elevated potassium",
        "rationale": "Patient takes lisinopril and spironolactone and has potassium 5.3 mmol/L.",
        "recommendation": "Flag baseline hyperkalemia risk and confirm potassium monitoring before adding medications that may worsen kidney or potassium status.",
        "evidence_ids": ["dailyMed_tmp_smx"],
    },
    {
        "pair": {"oxycodone", "alprazolam"},
        "severity": "high",
        "title": "Opioid and benzodiazepine overlap",
        "rationale": "Patient has active oxycodone and alprazolam prescriptions and obstructive sleep apnea.",
        "recommendation": "Review controlled-substance risk, sedation risk, naloxone need, and PDMP history if available.",
        "evidence_ids": ["cdc_opioid"],
    },
]

In [ ]:
def _canon(name: str) -> str:
    return name.lower().replace(",", " ").replace("  ", " ").strip()


def normalize_medication_name(name: str) -> dict:
    """Normalize a medication name to ingredients and classes using a local RxNorm-like map."""
    key = _canon(name)
    if key in DRUG_DB:
        return {"input": name, "normalized_name": key, **DRUG_DB[key]}

    if "sulfamethoxazole" in key or "trimethoprim" in key or "bactrim" in key:
        return {"input": name, "normalized_name": "sulfamethoxazole-trimethoprim ds", **DRUG_DB["sulfamethoxazole-trimethoprim ds"]}

    return {
        "input": name,
        "normalized_name": key,
        "ingredients": [key],
        "classes": [],
        "controlled": False,
    }


def patient_ingredients(patient: dict) -> list[str]:
    ingredients = []
    for med in patient["medications"]:
        normalized = normalize_medication_name(med["name"])
        ingredients.extend(normalized["ingredients"])
    return sorted(set(ingredients))


def patient_classes(patient: dict) -> list[str]:
    classes = []
    for med in patient["medications"]:
        normalized = normalize_medication_name(med["name"])
        classes.extend(normalized["classes"])
    return sorted(set(classes))


normalize_medication_name(proposed_rx.drug_name)

In [ ]:
try:
    from pydantic_ai import Tool
except Exception:
    def Tool(func=None):
        if func is None:
            return lambda f: f
        return func

    print("PydanticAI is not available yet. Using no-op Tool decorator for offline baseline.")


@Tool
def get_patient_context(patient_id: str) -> dict:
    """Return the synthetic FHIR-style patient context for a patient id."""
    return PATIENTS[patient_id]


@Tool
def normalize_medication(name: str) -> dict:
    """Normalize a medication name to ingredients, classes, and controlled-substance attributes."""
    return normalize_medication_name(name)


@Tool
def retrieve_evidence(evidence_ids: list[str]) -> list[dict]:
    """Return evidence citations for the requested evidence ids."""
    return [EVIDENCE_KB[eid] for eid in evidence_ids if eid in EVIDENCE_KB]

In [ ]:
def _finding_from_rule(rule: dict) -> Finding:
    return Finding(
        category="drug_interaction",
        severity=rule["severity"],
        title=rule["title"],
        rationale=rule["rationale"],
        recommendation=rule["recommendation"],
        evidence_ids=rule["evidence_ids"],
    )


def _severity_rank(severity: str) -> int:
    return {"low": 1, "moderate": 2, "high": 3, "critical": 4}[severity]


def _overall_risk(findings: list[Finding]) -> Severity:
    if not findings:
        return "low"
    return max((f.severity for f in findings), key=_severity_rank)


def _recommended_action(overall_risk: Severity) -> Action:
    return {
        "low": "approve",
        "moderate": "monitor",
        "high": "clarify",
        "critical": "block",
    }[overall_risk]


def run_deterministic_dur_review(patient_id: str, rx: ProposedPrescription) -> dict:
    patient = PATIENTS[patient_id]
    proposed = normalize_medication_name(rx.drug_name)
    active_ingredients = set(patient_ingredients(patient))
    proposed_ingredients = set(proposed["ingredients"])
    all_ingredients = active_ingredients | proposed_ingredients
    active_classes = set(patient_classes(patient))
    proposed_classes = set(proposed["classes"])

    findings: list[Finding] = []

    # Allergy review.
    for allergy in patient["allergies"]:
        allergy_text = allergy["substance"].lower()
        if "sulfonamide" in allergy_text and "sulfonamide_antibiotic" in proposed_classes:
            findings.append(
                Finding(
                    category="allergy",
                    severity="critical" if allergy["severity"] == "severe" else "high",
                    title="Sulfonamide antibiotic allergy conflict",
                    rationale=(
                        f"Patient has {allergy['severity']} allergy to {allergy['substance']} "
                        f"with reaction '{allergy['reaction']}', and the proposed medication contains sulfamethoxazole."
                    ),
                    recommendation="Block pending clinician/pharmacist review and clarify allergy history or select a non-sulfonamide alternative.",
                    evidence_ids=["dailyMed_tmp_smx"],
                )
            )

    # Drug-drug and background-risk review.
    for rule in INTERACTION_RULES:
        if rule["pair"].issubset(all_ingredients):
            findings.append(_finding_from_rule(rule))

    # Renal and lab monitoring review.
    egfr = patient["observations"].get("eGFR", {}).get("value")
    potassium = patient["observations"].get("potassium", {}).get("value")
    inr = patient["observations"].get("INR", {}).get("value")

    if egfr is None:
        findings.append(
            Finding(
                category="missing_data",
                severity="high",
                title="Missing kidney function",
                rationale="Renal function is required before reviewing dosing and monitoring for many antimicrobials.",
                recommendation="Obtain recent serum creatinine/eGFR before dispensing.",
                evidence_ids=[],
            )
        )
    elif egfr < 45 and proposed_ingredients.intersection({"sulfamethoxazole", "trimethoprim"}):
        findings.append(
            Finding(
                category="dose_adjustment",
                severity="moderate",
                title="Reduced kidney function requires dosing and monitoring review",
                rationale=f"Patient eGFR is {egfr} mL/min/1.73m2. TMP-SMX warrants renal function and potassium monitoring.",
                recommendation="Confirm renal dosing policy and monitoring plan before use.",
                evidence_ids=["dailyMed_tmp_smx"],
            )
        )

    if potassium is not None and potassium >= 5.0 and proposed_ingredients.intersection({"trimethoprim"}):
        findings.append(
            Finding(
                category="monitoring",
                severity="high",
                title="Elevated potassium before trimethoprim exposure",
                rationale=f"Patient potassium is {potassium} mmol/L and already has CKD plus ACE inhibitor/spironolactone exposure.",
                recommendation="Clarify whether TMP-SMX is necessary; consider alternative and repeat potassium monitoring.",
                evidence_ids=["dailyMed_tmp_smx"],
            )
        )

    if inr is not None and inr > 3.0 and proposed_ingredients.intersection({"sulfamethoxazole", "trimethoprim"}):
        findings.append(
            Finding(
                category="monitoring",
                severity="critical",
                title="Supratherapeutic INR before interacting antibiotic",
                rationale=f"Patient INR is {inr}. Adding TMP-SMX to warfarin may increase bleeding risk.",
                recommendation="Block pending anticoagulation plan, alternative antibiotic, or near-term INR monitoring.",
                evidence_ids=["dailyMed_tmp_smx", "dailyMed_warfarin"],
            )
        )

    evidence_ids = sorted({eid for finding in findings for eid in finding.evidence_ids})
    evidence = [EvidenceCitation(**EVIDENCE_KB[eid]) for eid in evidence_ids if eid in EVIDENCE_KB]
    overall = _overall_risk(findings)
    action = _recommended_action(overall)

    return ClinicalReviewDecision(
        patient_id=patient_id,
        prescription=rx,
        overall_risk=overall,
        recommended_action=action,
        key_findings=sorted(findings, key=lambda f: _severity_rank(f.severity), reverse=True),
        missing_information=[],
        evidence=evidence,
        clinician_message=(
            "High-risk prescription review: proposed TMP-SMX conflicts with documented sulfonamide allergy, "
            "active warfarin therapy with INR 3.2, CKD, and elevated potassium. Recommend blocking pending review "
            "and considering an alternative antibiotic or a documented monitoring plan."
        ),
        pharmacist_note=(
            "DUR: Do not dispense until prescriber clarification. Documented severe sulfonamide allergy; "
            "warfarin interaction with INR 3.2; CKD stage 3b with K 5.3 on lisinopril/spironolactone. "
            "Recommend alternative therapy or explicit INR/potassium monitoring plan."
        ),
    ).model_dump()


@Tool
def run_dur_review(patient_id: str, prescription: dict) -> dict:
    """Run deterministic DUR, allergy, interaction, renal, lab, and controlled-substance screening."""
    rx = ProposedPrescription(**prescription)
    return run_deterministic_dur_review(patient_id, rx)

Run the clinical review engine before adding the LLM. This gives you a deterministic baseline that can be evaluated and audited.

In [ ]:
try:
    from rich import print as rprint
except Exception:
    rprint = print

baseline_review = run_deterministic_dur_review("patient-001", proposed_rx)
rprint(baseline_review)

<a id="step7"></a>

## Step 7: Creating the MedGuard AI agent

The agent should not invent clinical facts. Its job is to call tools, preserve structured findings, add concise reasoning, and produce a pharmacist-ready intervention note.

In [ ]:
try:
    from pydantic_ai import Agent
except Exception as exc:
    Agent = None
    print("PydanticAI Agent is not available yet.")
    print(type(exc).__name__, str(exc)[:300])

system_prompt = """
You are MedGuard AI, a clinical prescription review copilot for synthetic US healthcare data.

Rules:
- You are clinical decision support, not an autonomous prescriber.
- Always call get_patient_context and run_dur_review before giving a recommendation.
- Preserve deterministic findings and severity from the DUR tool.
- Use retrieve_evidence for citation ids returned by findings.
- If data is missing, say what is missing.
- Keep the clinician message concise and actionable.
- Do not claim that FAERS or spontaneous adverse event reports prove causality.
"""

if Agent is None or agent_model is None:
    agent = None
    print("Agent skipped. The deterministic review engine remains available.")
else:
    agent_kwargs = dict(
        model=agent_model,
        tools=[
            get_patient_context,
            normalize_medication,
            retrieve_evidence,
            run_dur_review,
        ],
        system_prompt=system_prompt,
    )

    try:
        agent = Agent(**agent_kwargs, output_type=ClinicalReviewDecision)
        print("Agent created with typed output.")
    except TypeError:
        agent = Agent(**agent_kwargs)
        print("Agent created without typed-output constructor support. Prompt will request JSON output.")

<a id="step8"></a>

## Step 8: Running a clinical prescription review

If your vLLM server is running, this cell asks Qwen3 to orchestrate the tools and produce the final clinical decision. If the LLM endpoint is unavailable, use the deterministic `baseline_review` from the previous section for the demo.

In [ ]:
review_prompt = f"""
Review this proposed prescription for patient-001.

Proposed prescription:
{proposed_rx.model_dump()}

Return a structured clinical review with:
- overall_risk
- recommended_action
- key findings
- evidence citations
- clinician_message
- pharmacist_note
"""

try:
    llm_review = await run_async(review_prompt)
    rprint(llm_review)
except Exception as exc:
    print("LLM endpoint was not available or returned an error.")
    print("Using deterministic baseline review instead.")
    print(type(exc).__name__, str(exc)[:500])
    rprint(baseline_review)

You can also run a quick plain-English question against the same tools.

In [ ]:
question = "Should the pharmacist dispense TMP-SMX for patient-001 right now? Explain in 5 bullets with citations."

try:
    answer = await run_async(question)
    rprint(answer)
except Exception as exc:
    print("LLM endpoint unavailable. Baseline action:", baseline_review["recommended_action"])
    for finding in baseline_review["key_findings"][:5]:
        print("-", finding["severity"], finding["title"])

<a id="step9"></a>

## Step 9: Evaluating the solution

A competitive hackathon submission should show measurable value:

- **Clinical quality:** recall for critical findings, precision of actionable alerts, and gold-label agreement
- **Efficiency:** P50/P95 latency, tokens per review, and reviews per minute per GPU
- **Safety:** unsupported-claim rate, missing-data detection, citation coverage, and human-in-the-loop rate

In [ ]:
import time

GOLD = {
    "patient-001": {
        "required_titles": {
            "Sulfonamide antibiotic allergy conflict",
            "TMP-SMX may increase warfarin anticoagulation effect",
            "Supratherapeutic INR before interacting antibiotic",
        },
        "expected_action": "block",
    }
}


def evaluate_review(patient_id: str, review: dict) -> dict:
    titles = {finding["title"] for finding in review["key_findings"]}
    required = GOLD[patient_id]["required_titles"]
    found_required = titles.intersection(required)
    recall = len(found_required) / len(required)
    action_match = review["recommended_action"] == GOLD[patient_id]["expected_action"]
    citation_coverage = sum(1 for f in review["key_findings"] if f["evidence_ids"]) / max(1, len(review["key_findings"]))
    return {
        "required_finding_recall": round(recall, 3),
        "action_match": action_match,
        "citation_coverage": round(citation_coverage, 3),
        "overall_risk": review["overall_risk"],
        "recommended_action": review["recommended_action"],
    }


start = time.perf_counter()
review = run_deterministic_dur_review("patient-001", proposed_rx)
latency_ms = (time.perf_counter() - start) * 1000

metrics = evaluate_review("patient-001", review)
metrics["deterministic_latency_ms"] = round(latency_ms, 2)
metrics

## Enterprise architecture for the final pitch

For the hackathon presentation, position MedGuard AI as a FHIR-native, evidence-constrained review layer.

```text
EHR / Pharmacy System
     |
     v
FHIR Bundle + Proposed MedicationRequest
     |
     v
PydanticAI Orchestrator
     |
     +-- Medication Normalization Tool: RxNorm/RxNav
     +-- DUR Rules Tool: interactions, allergies, duplicate therapy
     +-- Patient Risk Tool: labs, age, renal/hepatic, pregnancy
     +-- Controlled Substance Tool: MME, opioid/benzo, PDMP stub
     +-- Evidence RAG Tool: DailyMed, labels, CDC guidance
     |
     v
Qwen3 on ROCm/vLLM
     |
     v
Structured ClinicalReviewDecision + Audit Trail
```

<a id="step10"></a>

## Step 10: Challenge ideas for the hackathon

Expand this prototype into a full enterprise-grade solution:

1. Replace the local drug map with live RxNorm/RxNav and DailyMed retrieval.
2. Generate 100 synthetic patients with Synthea and create gold-label test cases.
3. Add a vector database for labeling and guideline retrieval.
4. Add a UI with patient context, proposed prescription, findings, citations, and intervention note.
5. Add a controlled-substance dashboard with MME, opioid/benzodiazepine overlap, and PDMP simulation.
6. Add observability: latency, token usage, cache hits, severity distribution, and override reason.
7. Add a second Qwen3 "safety judge" agent that checks whether every clinical claim has evidence.

Final pitch line:

> MedGuard AI reduces alert fatigue by combining deterministic DUR screening, FHIR-native patient context, and evidence-grounded Qwen3 reasoning into a typed, auditable clinical review copilot.